# DDECC reproduction: clone → environment → check → train 14 codes → test 4/5/6 dB → plots

Paper: Choukroun & Wolf, *Denoising Diffusion Error Correction Codes*, ICLR 2023 — code: https://github.com/yoniLc/DDECC

**This notebook uses the results that are already on your Drive.** It reads every place your earlier runs wrote to:

| where | what |
|---|---|
| `DDECC_results/<CODE>__Code_n_.._k_..__<time>/` | trained models (`best_model`) + `logging.txt` with Main.py's own final test |
| `DDECC_results/<run>.log` | the old runner's copy of the console output |
| `DDECC_results/eval_official/log_<CODE>_<n>_<k>.txt` | resumable tests, regular `[reg]` and line search `[LS]` |
| `DDECC_results/eval_resume/eval_state_*.json` | resumable tests (regular, and the CCSDS line-search run) |

Anything this notebook adds goes into new folders, and nothing old is modified:
`DDECC_results/runs_v3/` (new training runs), `DDECC_results/eval_v3/` (new tests), `DDECC_results/figures_v3/` (tables and figures).

**Which steps need a GPU**

| step | GPU? |
|---|---|
| 1. Setup (mount Drive, settings, helper functions) | no, always run this first |
| 2. Environment: clone + patch the DDECC repo | yes (it also checks the GPU) |
| 3. Check: what is trained / tested | no |
| 4. Train the codes that have no model yet | yes |
| 5. Test the Eb/N0 points that have no result yet | yes |
| 6. Results: tables + figures | no |
| 7. Comparison with the DDECC paper | no |

With your current results, steps 3 → 6 → 7 show everything without a GPU (run step 1 first). Steps 4 and 5 find nothing to do unless you add codes to `RETRAIN` or delete results.
After a disconnect, run step 1 again and then the step you were in. Training and testing both continue where they stopped.

## 1. Setup: mount Drive, settings, helper functions (no GPU)

Every setting is in the first block of this cell. The rest only defines functions; it trains and tests nothing.

- **Training** = your earlier runs: 2000 epochs × 1000 batches, batch 128, lr 1e-4 (cosine to 5e-6), β = 0.01, N = 6, d = 128.
- **Testing** = the DDECC protocol: per Eb/N0, stop after > 500 frame errors and > 1e5 frames, or at 1e7 frames.
- `TEST_MODES`: `"reg"` = regular reverse diffusion, `"LS"` = line search (the paper's Table 1 uses LS).
- `STRICT_500`: your `eval_official` tests stopped at about 100 frame errors (Main.py's default), not the paper's 500. Those points are valid but less precise and are marked ✓¹. With `True`, step 5 tests them again to 500 errors (can take days in total). Default `False`.
- `RETRAIN`: codes to train again from scratch with another seed, e.g. `{("BCH", 63, 36): 1}`. The new model then replaces the old one in steps 3–7 and is tested again in step 5. The old model and its results stay on Drive. Empty by default.

In [ ]:
import os, sys, re, json, math, time, subprocess
from pathlib import Path
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# ============================== settings ==============================
ROOT  = Path("/content/drive/MyDrive/DDECC_results")    # your existing results folder
REPO  = Path("/content/DDECC")
CODES = [("BCH", 63, 36), ("BCH", 63, 45), ("BCH", 63, 51),
         ("POLAR", 64, 32), ("POLAR", 64, 48),
         ("POLAR", 128, 64), ("POLAR", 128, 86), ("POLAR", 128, 96),
         ("LDPC", 49, 24), ("LDPC", 121, 60), ("LDPC", 121, 70), ("LDPC", 121, 80),
         ("MACKAY", 96, 48), ("CCSDS", 128, 64)]
# training (same as your earlier runs)
N_DEC, D_MODEL, HEADS = 6, 128, 8
EPOCHS, BATCH_SIZE, LR, SIGMA = 2000, 128, 1e-4, 0.01
WORKERS, SAVE_EVERY = min(4, os.cpu_count() or 1), 50
RETRAIN = {}            # e.g. {("BCH", 63, 36): 1, ("POLAR", 128, 86): 1}  -> code: seed
# testing
EBNO = [4, 5, 6]
TEST_MODES = ["reg", "LS"]
MIN_FER, MIN_FRAMES, MAX_FRAMES = 500, 1e5, 1e7
STRICT_500 = False      # True: points that stopped with < 500 frame errors (old tests) are tested again in step 5
TEST_BATCH, SEED, SAVE_EVERY_S = 2048, 42, 120
# ======================================================================

RUNS, TESTS, FIGS = ROOT / "runs_v3", ROOT / "eval_v3", ROOT / "figures_v3"
for p in (RUNS, TESTS, FIGS):
    p.mkdir(parents=True, exist_ok=True)
MODE_NAME = {"reg": "Regular Reverse Diffusion", "LS": "Line Search Reverse Diffusion"}


def read(p):
    try:
        return Path(p).read_text(errors="ignore")
    except OSError:
        return ""


def read_int(p):
    try:
        return int(Path(p).read_text().strip())
    except (OSError, ValueError):
        return None


def nums(s):
    return [float(x) for x in re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s.replace("np.float64", ""))]


def max_epoch(d):
    eps = [int(x) for x in re.findall(r"Training epoch (\d+)", read(Path(d) / "logging.txt"))]
    return max(eps) if eps else 0


def name(c):
    return f"{c[0]}({c[1]},{c[2]})"


def folder_time(p):
    # run folders end with __DD_MM_YYYY_HH_MM_SS: order by that time, not by Drive's file dates
    m = re.search(r"__(\d\d)_(\d\d)_(\d{4})_(\d\d)_(\d\d)_(\d\d)$", Path(p).name)
    if m:
        d, mo, y, h, mi, s = map(int, m.groups())
        return (y, mo, d, h, mi, s)
    return (0,) * 6


def tag(c):
    return f"{c[0]}_{c[1]}_{c[2]}"


def run_dir(c, seed):
    return RUNS / f"{c[0]}_n{c[1]}_k{c[2]}_N{N_DEC}_d{D_MODEL}_seed{seed}"


def save_json(path, obj):
    for _ in range(10):                      # Drive can fail for a moment: retry
        try:
            tmp = Path(str(path) + ".tmp")
            tmp.write_text(json.dumps(obj, indent=1))
            os.replace(tmp, path)
            return
        except OSError as e:
            print("save failed, retrying:", e, flush=True)
            time.sleep(10)


# ---------------------------- which model ----------------------------
def find_model(c):
    # -> dict(state=trained|resume|fresh, model=..., src=new|old, dir=..., seed=..)
    t, n, k = c
    seed = RETRAIN.get(c, 42)
    d = run_dir(c, seed)
    e = dict(dir=str(d), seed=seed)
    if (d / "train_done.txt").exists() and (d / "best_model").exists():
        return {**e, "state": "trained", "model": str(d / "best_model"), "src": "new"}
    old = sorted(ROOT.glob(f"{t}__Code_n_{n}_k_{k}__*"), key=folder_time)
    if c not in RETRAIN:
        done = [x for x in old if (x / "best_model").exists() and max_epoch(x) >= EPOCHS]
        if done:
            return {**e, "state": "trained", "model": str(done[-1] / "best_model"), "src": "old"}
    cands = []
    for x in [d] + ([] if c in RETRAIN else old):
        ep = read_int(x / "last_epoch.txt")
        if ep and (x / "last_model").exists():
            cands.append((ep, str(x / "last_model")))
    if cands:
        ep, path = max(cands)
        return {**e, "state": "resume", "resume": path, "start": ep + 1}
    return {**e, "state": "fresh"}


# ---------------------------- results ----------------------------
PROG = re.compile(r"\[(reg|LS)\] EbN0=(\d+): ([\d.eE+]+) frames, (\d+) frame errors, (\d+) bit errors")


def res_official(c):
    t, n, k = c
    out = {}
    for line in read(ROOT / "eval_official" / f"log_{tag(c)}.txt").splitlines():
        m = PROG.search(line)
        if not m or int(m[2]) not in EBNO:
            continue
        mode, e, fr, fe, be = m[1], int(m[2]), float(m[3]), int(m[4]), int(m[5])
        done = line.rstrip().endswith("done")
        old = out.get((mode, e))
        if old and old["done"] and not done:
            continue
        out[(mode, e)] = dict(ber=be / (fr * n), fer=fe / fr, frames=fr, fe=fe, done=done,
                              src=f"eval_official/log_{tag(c)}.txt")
    return out


def res_eval_resume(c, model):
    t, n, k = c
    out = {}
    for f, mode in [(f"eval_state_{tag(c)}.json", "reg"), (f"eval_state_{tag(c)}_reg.json", "reg"),
                    (f"eval_state_{tag(c)}_LS.json", "LS")]:
        s = json.loads(read(ROOT / "eval_resume" / f) or "{}")
        if s.get("meta", {}).get("model_path") != model:
            continue
        for e, st in s.items():
            if e != "meta" and int(e) in EBNO and st.get("bit_err"):
                fr = st["frames"]
                out.setdefault((mode, int(e)), dict(ber=st["bit_err"][-1] / (fr * n), fer=st["frame_err"][-1] / fr,
                                                    frames=fr, fe=st["frame_err"][-1], done=bool(st.get("done")),
                                                    src=f"eval_resume/{f}"))
    return out


def _main_section(txt, section):
    i = txt.rfind(section)
    if i < 0:
        return {}
    seg = txt[i + len(section):]
    for other in MODE_NAME.values():
        j = seg.find(other)
        if j >= 0:
            seg = seg[:j]
    m = re.search(r"# of testing samples: \[([^\]]*)\]", seg)
    frames = nums(m.group(1)) if m else []
    out = {}
    for idx, e in enumerate([4, 5, 6]):          # Main.py tests range(4, 7)
        mb = re.search(rf"Test EbN0={e}, BER=\[([^\]]*)\]", seg)
        mf = re.search(rf"Test EbN0={e}, FER_DDPM=\[([^\]]*)\]", seg)
        if mb and mf and e in EBNO:
            fr = frames[idx] if idx < len(frames) else None
            fer = nums(mf.group(1))[-1]
            out[e] = dict(ber=nums(mb.group(1))[-1], fer=fer, frames=fr, fe=round(fer * fr) if fr else None, done=True)
    return out


def res_main_logs(c, model):
    t, n, k = c
    mdir = Path(model).parent
    texts = [(read(mdir / "logging.txt"), mdir.name + "/logging.txt")]
    for d in sorted(ROOT.glob(f"{t}__Code_n_{n}_k_{k}__*"), key=folder_time, reverse=True):
        if d != mdir and f"resume='{model}'" in read(d / "logging.txt"):
            texts.append((read(d / "logging.txt"), d.name + "/logging.txt"))   # old "test only" runs
    for lf in ROOT.glob(f"{t}_n{n}_k{k}_N*_d*.log"):
        for part in reversed(re.split(r"(?m)^(?=Path to model/logs:)", read(lf))):
            first = part.split("\n", 1)[0]
            if Path(first.split(":", 1)[-1].strip()).name == mdir.name or f"resume='{model}'" in part:
                texts.append((part, lf.name))
    out = {}
    for txt, src in texts:
        for mode, sec in MODE_NAME.items():
            for e, v in _main_section(txt, sec).items():
                v = {**v, "src": src}
                if (mode, e) not in out or rank(v) > rank(out[(mode, e)]):
                    out[(mode, e)] = v
    return out


def v3_file(model, mode):
    return TESTS / f"{Path(model).parent.name}_{mode}.json"


def res_v3(c, model):
    t, n, k = c
    out = {}
    for mode in MODE_NAME:
        s = json.loads(read(v3_file(model, mode)) or "{}")
        if s.get("meta", {}).get("model_path") != model:
            continue
        for e, st in s.items():
            if e != "meta" and int(e) in EBNO and st.get("bit_err"):
                fr = st["frames"]
                out[(mode, int(e))] = dict(ber=st["bit_err"][-1] / (fr * n), fer=st["frame_err"][-1] / fr,
                                           frames=fr, fe=st["frame_err"][-1], done=bool(st.get("done")),
                                           src=f"eval_v3/{v3_file(model, mode).name}")
    return out


def results(c, m):
    # every (mode, Eb/N0) -> result of THIS model; a finished one beats an unfinished one
    if m["state"] != "trained":
        return {}
    model = m["model"]
    sources = [res_v3(c, model)]
    if m["src"] == "old":                    # the old result files belong to the old models
        sources += [res_official(c), res_eval_resume(c, model), res_main_logs(c, model)]
    out = {}
    for src in sources:
        for key, v in src.items():
            if key not in out or rank(v) > rank(out[key]):
                out[key] = v
    return out


def protocol_ok(r):
    # DDECC paper: >= 500 frame errors (and >= 1e5 frames); the 1e7-frame cap ends a point too.
    # None = unknown (an old Main.py log without the frame count)
    if not r["done"]:
        return False
    if r["fe"] is None and r["frames"] is None:
        return None
    return (r["fe"] or 0) >= MIN_FER or (r["frames"] or 0) >= MAX_FRAMES


def rank(r):
    # when one point was tested several times: finished > unfinished, then the protocol-compliant
    # one (known > unknown > below 500 errors), then the one with more frame errors
    ok = protocol_ok(r)
    return (r["done"], 2 if ok else 1 if ok is None else 0, r["fe"] or 0, r["frames"] or 0)


def is_done(r):
    return bool(r) and r["done"] and (protocol_ok(r) is not False if STRICT_500 else True)


def mark(r):
    # ✓ finished; ✓? finished, frame count unknown; ✓¹ finished with < 500 errors; … unfinished; — none
    if not r:
        return "—"
    if not r["done"]:
        return "…"
    ok = protocol_ok(r)
    return "✓" if ok else "✓?" if ok is None else "✓¹"


def check(verbose=True):
    S = {}
    for c in CODES:
        m = find_model(c)
        S[c] = dict(model=m, res=results(c, m))
    if verbose:
        head = "".join(f"{m}{e}".rjust(7) for m in TEST_MODES for e in EBNO)
        print(f"{'code':<15}{'training':<24}{head}   model")
        print("-" * 118)
        for c, s in S.items():
            m = s["model"]
            st = {"trained": "✅ trained" + (" (old)" if m.get("src") == "old" else " (new)"),
                  "resume": f"⏸ resume {m.get('start')}→{EPOCHS}", "fresh": "⬜ not trained"}[m["state"]]
            cells = ""
            for mode in TEST_MODES:
                for e in EBNO:
                    r = s["res"].get((mode, e))
                    cells += mark(r).rjust(7)
            where = (m.get("model") or m.get("resume") or m["dir"]).replace(str(ROOT) + "/", "")
            print(f"{name(c):<15}{st:<24}{cells}   {where}")
        n_tr = sum(s["model"]["state"] == "trained" for s in S.values())
        n_te = sum(all(is_done(s["res"].get((mo, e))) for mo in TEST_MODES for e in EBNO) for s in S.values())
        print("-" * 118)
        print(f"trained {n_tr}/{len(S)}, fully tested {n_te}/{len(S)}   (✓ finished, … unfinished, — no result)")
        print(f"✓¹ finished with < {MIN_FER} frame errors before {MAX_FRAMES:.0e} frames (an old test that stopped at Main.py's "
              f"100 errors): valid but less precise; STRICT_500 = True re-tests these in step 5")
        print("✓? finished, but the old Main.py log does not record the frame count")
    return S


def lnb(r):
    return -math.log(r["ber"]) if r["ber"] > 0 else float("inf")


# ---------------------------- DDECC repo + patches ----------------------------
def rep(src, old, new, what):
    k = src.count(old)
    assert k == 1, f"patch '{what}' failed: the original text occurs {k} times (expected 1)"
    return src.replace(old, new)


def ensure_repo():
    # clone the DDECC repo; restore Main.py/DDECC.py and patch them again (safe to repeat)
    if not (REPO / "Main.py").exists():
        subprocess.run(["git", "clone", "-q", "https://github.com/yoniLc/DDECC.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--", "Main.py", "DDECC.py"], check=True)
    src = (REPO / "Main.py").read_text()
    src = rep(src, "def main(args):", '''def _safe_save(obj, path, retries=10, wait=30):
    for i in range(retries):            # temp file + rename; retry on short Drive outages
        try:
            torch.save(obj, path + '.tmp')
            os.replace(path + '.tmp', path)
            return True
        except Exception as e:
            print(f'[save] {path} failed ({e}); retry {i + 1}/{retries}', flush=True)
            time.sleep(wait)
    return False


def _safe_write(path, text, retries=10, wait=30):
    for i in range(retries):
        try:
            with open(path + '.tmp', 'w') as f:
                f.write(text)
            os.replace(path + '.tmp', path)
            return True
        except Exception as e:
            print(f'[write] {path} failed ({e}); retry {i + 1}/{retries}', flush=True)
            time.sleep(wait)
    return False


def main(args):''', "helpers")
    src = rep(src, "    parser.add_argument('--sigma', type=float, default=0.01)\n",
              "    parser.add_argument('--sigma', type=float, default=0.01)\n"
              "    parser.add_argument('--out_dir', type=str, default='')\n"
              "    parser.add_argument('--resume', type=str, default='')\n"
              "    parser.add_argument('--start_epoch', type=int, default=1)\n"
              "    parser.add_argument('--save_every', type=int, default=50)\n"
              "    parser.add_argument('--no_test', action='store_true')\n", "arguments")
    src = rep(src, "    os.makedirs(model_dir, exist_ok=True)\n",
              "    if args.out_dir:\n        model_dir = args.out_dir\n    os.makedirs(model_dir, exist_ok=True)\n", "out_dir")
    src = rep(src, "    model = DDECCT(args, device=device,dropout=0).to(device)\n    model.ema.register(model)\n",
              "    if args.resume:\n"
              "        model = torch.load(args.resume, map_location=device, weights_only=False).to(device)\n"
              "        model.device = device\n"
              "        logging.info(f'Resumed from {args.resume}, start_epoch={args.start_epoch}')\n"
              "    else:\n"
              "        model = DDECCT(args, device=device,dropout=0).to(device)\n"
              "        model.ema.register(model)\n", "model creation")
    src = rep(src, "    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=5e-6)\n",
              "    scheduler = CosineAnnealingLR(optimizer, T_max=args.epochs, eta_min=5e-6)\n"
              "    for _ in range(args.start_epoch - 1):\n"
              "        scheduler.step()\n"
              "    _opt = os.path.join(os.path.dirname(args.resume), 'last_optim.pt') if args.resume else ''\n"
              "    if _opt and os.path.exists(_opt):\n"
              "        _cur = optimizer.state_dict()\n"
              "        _cur['state'] = torch.load(_opt, map_location=device, weights_only=False)['state']\n"
              "        optimizer.load_state_dict(_cur)\n"
              "        logging.info('Adam state restored')\n", "scheduler")
    src = rep(src, "    best_loss = float('inf')\n    for epoch in range(1, args.epochs + 1):\n",
              "    best_loss = float('inf')\n"
              "    _bl = os.path.join(os.path.dirname(args.resume), 'best_loss.txt') if args.resume else ''\n"
              "    if _bl and os.path.exists(_bl):\n"
              "        best_loss = float(open(_bl).read())\n"
              "    for epoch in range(args.start_epoch, args.epochs + 1):\n", "epoch loop")
    src = rep(src, "            torch.save(model, os.path.join(args.path, 'best_model'))\n            logging.info(f'Model Saved')\n",
              "            if _safe_save(model, os.path.join(args.path, 'best_model')):\n"
              "                _safe_write(os.path.join(args.path, 'best_loss.txt'), repr(best_loss))\n"
              "            logging.info(f'Model Saved')\n"
              "        if epoch % args.save_every == 0 or epoch == args.epochs:\n"
              "            if (_safe_save(model, os.path.join(args.path, 'last_model')) and\n"
              "                    _safe_save(optimizer.state_dict(), os.path.join(args.path, 'last_optim.pt'))):\n"
              "                _safe_write(os.path.join(args.path, 'last_epoch.txt'), str(epoch))\n", "save")
    src = rep(src, "        if epoch % (args.epochs//2) == 0 or epoch in [1,25]:\n",
              "        if (not args.no_test) and (epoch % (args.epochs//2) == 0 or epoch in [1,25]):\n", "mid test")
    src = rep(src, "    logging.info('Loading Best Model')\n    model = torch.load(os.path.join(args.path, 'best_model')).to(device)\n",
              "    _safe_write(os.path.join(args.path, 'train_done.txt'), str(args.epochs))\n"
              "    if args.no_test:\n"
              "        logging.info('Training finished (testing is done by the notebook)')\n"
              "        return\n"
              "    logging.info('Loading Best Model')\n"
              "    model = torch.load(os.path.join(args.path, 'best_model'), weights_only=False).to(device)\n", "final test")
    compile(src, "Main.py", "exec")
    (REPO / "Main.py").write_text(src)
    dsrc = (REPO / "DDECC.py").read_text()
    dsrc = rep(dsrc, "        #Single sampling from the real p dist.\n",
               "        #Single sampling from the real p dist.\n        yt = yt.cpu()  # [patch] reverse step on the CPU\n", "yt cpu")
    dsrc = rep(dsrc, "        t = sum_syndrome\n", "        t = sum_syndrome.cpu()  # [patch] CPU index for the CPU betas\n", "t cpu")
    compile(dsrc, "DDECC.py", "exec")
    (REPO / "DDECC.py").write_text(dsrc)


SETUP_OK = True
print("Setup done. Results folder:", ROOT, "(found)" if ROOT.exists() else "(NOT FOUND)")

## 2. Environment: clone + patch the DDECC repo, check the GPU (GPU)

Clones https://github.com/yoniLc/DDECC to `/content/DDECC`. Each run first restores `Main.py` / `DDECC.py` to the original version and then patches them again, so the patches are never applied twice. Every patch checks that the text it replaces occurs exactly once.

**Patches to Main.py** (training only, the model and the loss are unchanged):
- `--out_dir`: one fixed folder per run.
- `--resume` / `--start_epoch`: continue from `last_model`; the cosine lr is fast-forwarded, and the Adam state and `best_loss` are restored.
- `last_model` every 50 epochs, safe saves (temp file + rename, retries on Drive outages), `train_done.txt` at the end.
- `--no_test`: no tests inside Main.py; step 5 tests instead, and can resume.
- `torch.load(..., weights_only=False)` for PyTorch ≥ 2.6.

**Patch to DDECC.py:** one reverse step (`p_sample`) runs on the CPU, so the CPU `betas` are never indexed with a GPU tensor (the device error that crashed an earlier BCH test).

Steps 4 and 5 call the same function themselves, so this cell is optional. It is useful to check the GPU before a long run.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

ensure_repo()
import torch
print("DDECC repo cloned and patched ✓")
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE -> Runtime → Change runtime type → GPU")

## 3. Check: what is trained, what is tested (no GPU)

For each code:
- **training**: ✅ trained (old = your latest earlier folder, by the date in its name, that reaches epoch 2000; new = trained by this notebook), ⏸ can resume, ⬜ not trained.
- **one column per decoding × Eb/N0** (`reg4 … LS6`): ✓ finished, … unfinished (step 5 continues it), — no result (step 5 tests it).
- **model**: the `best_model` that all results belong to.

Results are read from every file listed at the top. When the same point was tested more than once, the finished one is used, then the one meeting the paper's rule (≥ 500 frame errors, or 1e7 frames), then the one with more frame errors. Results of an old model are never used for a new one.

✓¹ = finished, but with fewer than 500 frame errors before 1e7 frames (the old tests stopped at about 100 errors). Valid, but less precise; see `STRICT_500` in step 1. ✓? = finished, but the old Main.py log does not record the frame count.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

S = check()

## 4. Train the codes that have no model yet (GPU)

Runs the patched `Main.py` for every code that step 3 shows as ⏸ or ⬜, or that is listed in `RETRAIN`:
- ⬜ from scratch: epoch 1 → 2000, output in `DDECC_results/runs_v3/<code>_seed<seed>/`.
- ⏸ resume: from the next epoch after `last_model` (lr, Adam state and `best_loss` restored).

The notebook prints one line every 10 epochs; the full log is in `logging.txt` of the run folder.
After a disconnect: run step 1, then this step. It continues from the last checkpoint (at most 50 epochs are lost).
With your current results there is nothing to train, and the cell says so.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

from IPython.display import clear_output
S = check(verbose=False)
todo = [c for c, s in S.items() if s["model"]["state"] != "trained"]
if not todo:
    print("Nothing to train: all codes have a trained model.")
else:
    ensure_repo()
    import torch
    assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
    failed = []
    for i, c in enumerate(todo, 1):
        m = S[c]["model"]
        cmd = [sys.executable, "-u", "Main.py", "--gpus=0", f"--code_type={c[0]}", f"--code_n={c[1]}", f"--code_k={c[2]}",
               f"--N_dec={N_DEC}", f"--d_model={D_MODEL}", f"--h={HEADS}", f"--lr={LR}", f"--batch_size={BATCH_SIZE}",
               f"--test_batch_size={TEST_BATCH}", f"--epochs={EPOCHS}", f"--sigma={SIGMA}", f"--workers={WORKERS}",
               f"--seed={m['seed']}", f"--out_dir={m['dir']}", f"--save_every={SAVE_EVERY}", "--no_test"]
        start = 1
        if m["state"] == "resume":
            start = m["start"]
            cmd += [f"--resume={m['resume']}", f"--start_epoch={start}"]
        clear_output(wait=True)
        print(f"[{i}/{len(todo)}] {name(c)}  epoch {start} → {EPOCHS}  seed {m['seed']}\nlog: {m['dir']}/logging.txt", flush=True)
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        tail = []
        for line in proc.stdout:
            tail = (tail + [line])[-30:]
            mm = re.match(r"Training epoch (\d+), Batch (\d+)/(\d+)", line)
            if mm:
                if mm[2] == mm[3] and int(mm[1]) % 10 == 0:
                    print(time.strftime("[%m-%d %H:%M] ") + line.strip(), flush=True)
            elif line.strip() and not line.startswith(("Epoch ", "Model Saved", " ", "DDECCT(", ")")):
                print(line, end="", flush=True)
        proc.wait()
        if proc.returncode != 0:
            failed.append(name(c))
            print(f"\n❌ {name(c)} failed (exit code {proc.returncode}); last output:\n" + "".join(tail))
        else:
            print(f"\n✅ {name(c)} trained", flush=True)
    clear_output(wait=True)
    if failed:
        print("Failed:", failed)
S = check()

## 5. Test the Eb/N0 points that have no result yet (GPU)

For every trained model, every decoding in `TEST_MODES` and every Eb/N0 shown as — or … in step 3: random codewords → AWGN → DDECC reverse diffusion → bit and frame errors. The counting is the same as `test()` in the repo's `Main.py`, using the output of the last step.

- **Line search** is DDECC.py's own procedure (each step tries step sizes 1…20 and keeps the one with the fewest unsatisfied checks). This version was checked to give exactly the same output as the original code.
- **Saving**: progress is saved every 2 minutes to `DDECC_results/eval_v3/<model folder>_<reg|LS>.json`. After a disconnect, run step 1 and this step: finished points are skipped and an unfinished one continues from its saved counts. A batch's random numbers depend only on (seed, Eb/N0, batch number), so a resumed test gives the same result as an uninterrupted one.
- **Time**: at 6 dB a point often runs to 1e7 frames, which can take more than ten hours for the larger codes.
- **Stopping by hand**: press stop and the progress is saved first.

With your current results there is nothing to test, and the cell says so.

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

S = check(verbose=False)
jobs = [(c, mode, e) for c, s in S.items() if s["model"]["state"] == "trained"
        for mode in TEST_MODES for e in EBNO if not is_done(s["res"].get((mode, e)))]
if not jobs:
    print("Nothing to test: every point of every trained model is finished.")
else:
    ensure_repo()
    import torch
    assert torch.cuda.is_available(), "Runtime → Change runtime type → GPU"
    device = torch.device("cuda")
    sys.path.insert(0, str(REPO)); os.chdir(REPO)
    from Codes import Get_Generator_and_Parity, EbN0_to_std, sign_to_bin, bin_to_sign
    import DDECC

    def p_sample(self, yt):
        # DDECC.py's p_sample (regular or line search), indexing on the CPU, line search on the GPU
        yt = yt.cpu()
        sum_syndrome = (torch.matmul(sign_to_bin(torch.sign(yt.to(self.device))), self.pc_matrix) % 2).round().long().sum(-1)
        t = sum_syndrome.cpu()
        noise_mul_pred = self(yt.to(self.device), sum_syndrome.to(self.device)).cpu()
        noise_add_pred = yt - torch.sign(-noise_mul_pred * torch.sign(yt))
        bb, b = self.betas_bar.cpu(), self.betas.cpu()
        factor = torch.sqrt(bb[t]) * b[t] / (bb[t] + b[t])
        alpha_final = 1
        if self.line_search:
            alpha = torch.linspace(1, 20, 20, device=self.device).view(1, 1, -1)
            step = (noise_add_pred * factor).to(self.device).unsqueeze(-1)
            cand = sign_to_bin(torch.sign(yt.to(self.device).unsqueeze(-1) - alpha * step)).permute(0, 2, 1)
            new_synd = (torch.matmul(cand, self.pc_matrix) % 2).round().long().sum(-1)
            alpha_final = alpha.view(-1)[new_synd.argmin(-1)].cpu().unsqueeze(-1)
        yt_1 = yt - alpha_final * noise_add_pred * factor
        yt_1[t == 0] = yt[t == 0]
        return yt_1, t

    DDECC.DDECCT.p_sample = p_sample

    class Code:                                   # the saved models refer to __main__.Code
        pass

    def log(msg):
        print(time.strftime("[%m-%d %H:%M:%S] ") + msg, flush=True)

    print(f"{len(jobs)} point(s) to test:", ", ".join(f"{name(c)} {m} {e}dB" for c, m, e in jobs))
    loaded = None
    for c, mode, ebno in jobs:
        t, n, k = c
        model_path = S[c]["model"]["model"]
        if loaded != (c, model_path):
            torch.cuda.empty_cache()
            code_obj = Code(); code_obj.n, code_obj.k, code_obj.code_type = n, k, t
            G = torch.from_numpy(Get_Generator_and_Parity(code_obj)[0]).long()
            model = torch.load(model_path, map_location=device, weights_only=False)
            model.device = device; model.eval()
            loaded = (c, model_path)
        model.line_search = (mode == "LS")
        f = v3_file(model_path, mode)
        meta = dict(model_path=model_path, decoding=mode, min_fer=MIN_FER, min_frames=MIN_FRAMES,
                    max_frames=MAX_FRAMES, test_batch=TEST_BATCH, seed=SEED)
        state = json.loads(read(f) or "{}") or {"meta": meta}
        if state.get("meta") != meta:
            f.rename(f.with_suffix(f".bak{int(time.time())}")); state = {"meta": meta}
            log(f"{f.name}: other settings -> old file renamed to .bak, starting over")
        st = state.setdefault(str(ebno), dict(frames=0, batches=0, bit_err=None, frame_err=None, seconds=0.0, done=False))
        log(f"===== {name(c)} [{mode}] {ebno} dB" + (f", resuming at {st['frames']:.3e} frames" if st["frames"] else ""))
        std = EbN0_to_std(ebno, k / n)
        t0 = last = start = time.time(); fr0 = st["frames"]
        try:
            while True:
                gen = torch.Generator().manual_seed(SEED * 1000003 + ebno * 10007 + st["batches"])
                x = (torch.randint(0, 2, (TEST_BATCH, k), generator=gen) @ G) % 2
                y = (bin_to_sign(x) + torch.randn(TEST_BATCH, n, generator=gen) * std).float()
                with torch.no_grad():
                    _, res, _, _ = model.p_sample_loop(y)
                if st["bit_err"] is None:
                    st["bit_err"], st["frame_err"] = [0] * len(res), [0] * len(res)
                for kk, y_k in enumerate(res):
                    err = sign_to_bin(torch.sign(y_k)) != x.float()
                    st["bit_err"][kk] += int(err.sum()); st["frame_err"][kk] += int(err.any(dim=1).sum())
                st["frames"] += TEST_BATCH; st["batches"] += 1
                fe, be, fr = st["frame_err"][-1], st["bit_err"][-1], st["frames"]
                finished = (fe > MIN_FER and fr > MIN_FRAMES) or fr >= MAX_FRAMES
                if finished or time.time() - last > SAVE_EVERY_S:
                    st["seconds"] += time.time() - t0; t0 = last = time.time()
                    st["done"] = finished
                    save_json(f, state)
                    rate = (fr - fr0) / max(time.time() - start, 1e-9)
                    l = f"{-math.log(be / (fr * n)):.3f}" if be else "inf"
                    log(f"  [{mode}] EbN0={ebno}: {fr:.3e} frames, {fe} frame errors, {be} bit errors, -ln(BER)={l}, "
                        f"{rate:.0f} frames/s, " + ("done" if finished else f"at most {(MAX_FRAMES - fr) / max(rate, 1e-9) / 3600:.1f} h left"))
                if finished:
                    break
        except KeyboardInterrupt:
            st["seconds"] += time.time() - t0; save_json(f, state)
            log("stopped by hand, progress saved; run step 1 and this step again to continue")
            raise
S = check()

## 6. Results: tables + figures (no GPU)

1. Tables of −ln(BER) (higher is better) for regular and line-search decoding: 14 codes × 4/5/6 dB. `*` = unfinished point (its value so far).
2. Where every point comes from.
3. `results.csv` with BER, FER, frames, frame errors and the source of every point.
4. Figures for each decoding: −ln(BER) and BER (log scale) vs Eb/N0, one panel per code family. Plus one table figure with all 14 codes × reg/LS × 4/5/6 dB.

Everything is saved to `DDECC_results/figures_v3/` (PNG, 300 dpi).

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

S = check(verbose=False)
R = {c: s["res"] for c, s in S.items()}

for mode in TEST_MODES:
    print("\n" + "=" * 62 + f"\n−ln(BER), {MODE_NAME[mode]}   (* unfinished, ¹ < {MIN_FER} frame errors, ? frame count unknown, — no result)")
    print(f"{'code':<16}" + "".join(f"{e} dB".rjust(10) for e in EBNO))
    for c in CODES:
        row = ""
        for e in EBNO:
            r = R[c].get((mode, e))
            tail = {"—": "", "…": " *", "✓": "  ", "✓?": "? ", "✓¹": "¹ "}[mark(r)]
            row += "—".rjust(10) if not r else f"{lnb(r):8.2f}{tail}"
        print(f"{name(c):<16}{row}")

print("\nsources:")
for c in CODES:
    print(f"  {name(c):<15}", "; ".join(sorted({r['src'] for r in R[c].values()})) or "no results")

rows = ["code,n,k,decoding,ebno,neg_ln_ber,ber,fer,frames,frame_errors,finished,model,source"]
for c in CODES:
    for (mode, e), r in sorted(R[c].items()):
        rows.append(f"{c[0]},{c[1]},{c[2]},{mode},{e},{lnb(r):.4f},{r['ber']:.6e},{r['fer']:.6e},{r['frames'] or ''},"
                    f"{'' if r['fe'] is None else r['fe']},{r['done']},{S[c]['model'].get('model', '')},{r['src']}")
(FIGS / "results.csv").write_text("\n".join(rows) + "\n")

COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]     # fixed order, one per code
MARKERS = ["o", "s", "^", "D", "v"]
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
mpl.rcParams.update({"font.size": 10, "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.titlecolor": INK, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8, "legend.frameon": False})
FAMILIES = ["BCH", "POLAR", "LDPC", "MACKAY", "CCSDS"]

for mode in TEST_MODES:
    for ycol, ylabel, logy in [("lnb", "−ln(BER)  (higher is better)", False), ("ber", "BER", True)]:
        fig, axes = plt.subplots(1, 5, figsize=(18, 3.9), constrained_layout=True)
        for ax, fam in zip(axes, FAMILIES):
            for j, c in enumerate([c for c in CODES if c[0] == fam]):
                pts = [(e, R[c][(mode, e)]) for e in EBNO if (mode, e) in R[c] and R[c][(mode, e)]["ber"] > 0]
                if pts:
                    ax.plot([e for e, _ in pts], [lnb(r) if ycol == "lnb" else r["ber"] for _, r in pts],
                            color=COLORS[j], marker=MARKERS[j], lw=2, ms=7, markeredgecolor="white",
                            markeredgewidth=1.2, label=f"({c[1]},{c[2]})")
            if ax.lines:
                ax.legend(fontsize=9)
            else:
                ax.text(0.5, 0.5, "no data", transform=ax.transAxes, ha="center", color=INK2)
            ax.set_title(fam, fontweight="bold", loc="left")
            ax.set_xlim(min(EBNO) - 0.2, max(EBNO) + 0.2); ax.set_xticks(EBNO); ax.set_xlabel("Eb/N0 (dB)")
            if logy:
                ax.set_yscale("log")
        axes[0].set_ylabel(ylabel)
        fig.suptitle(f"DDECC reproduction (N={N_DEC}, d={D_MODEL}) — {ylabel.split()[0]} vs Eb/N0, {MODE_NAME[mode]}",
                     x=0.01, ha="left", fontsize=12)
        fig.savefig(FIGS / f"{ycol}_{mode}.png", dpi=300, bbox_inches="tight")
        plt.show()

cols = [(m, e) for m in TEST_MODES for e in EBNO]
vals = np.array([[lnb(R[c][key]) if key in R[c] else np.nan for key in cols] for c in CODES])
fin = vals[np.isfinite(vals)]
fig, ax = plt.subplots(figsize=(1.4 * len(cols) + 2.6, 0.42 * len(CODES) + 1.4), constrained_layout=True)
cmap = mpl.colors.LinearSegmentedColormap.from_list("blue", ["#cde2fb", "#86b6ef", "#2a78d6", "#184f95"])
ax.imshow(np.where(np.isfinite(vals), vals, np.nan), cmap=cmap, aspect="auto",
          vmin=fin.min() if fin.size else 0, vmax=fin.max() if fin.size else 1)
for i, c in enumerate(CODES):
    for j, key in enumerate(cols):
        v, r = vals[i, j], R[c].get(key)
        txt = "—" if np.isnan(v) else ("inf" if np.isinf(v) else f"{v:.2f}") + ("" if r is None or r["done"] else "*")
        dark = np.isfinite(v) and fin.size and v > fin.min() + 0.55 * (fin.max() - fin.min())
        ax.text(j, i, txt, ha="center", va="center", fontsize=9, color="white" if dark else INK)
ax.set_xticks(range(len(cols)), [f"{m}\n{e} dB" for m, e in cols])
ax.set_yticks(range(len(CODES)), [name(c) for c in CODES])
for x in range(len(EBNO), len(cols), len(EBNO)):
    ax.axvline(x - 0.5, color="white", lw=3)
ax.tick_params(length=0); ax.grid(False); ax.xaxis.tick_top()
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title("−ln(BER)   (reg = regular, LS = line search; * = unfinished)", loc="left", fontsize=10, pad=34)
fig.savefig(FIGS / "table_neglnBER.png", dpi=300, bbox_inches="tight")
plt.show()
print("saved to", FIGS, ":", ", ".join(sorted(p.name for p in FIGS.iterdir())))

## 7. Comparison with the DDECC paper (no GPU)

Compares each decoding with the paper's setting N = 6, d = 128:
- **LS**: Table 1 of the paper (the table's results use the line search). Each cell of that table holds d = 32 / 64 / 128; the d = 128 value is used.
- **Regular**: the paper gives no regular table, but Appendix E (Table 4) gives Δ = −ln(BER_LS) + ln(BER_reg) for all 14 codes. So paper regular = Table 1 − Δ. Both are rounded to 0.01, so the derived values are ±0.01.

Output: a table of differences (reproduction − paper; negative = reproduction lower) and one figure per decoding (blue = reproduction, orange dashed = paper, 14 panels).

In [ ]:
if "SETUP_OK" not in globals():
    raise SystemExit("Run step 1 first (it needs no GPU).")

import matplotlib.pyplot as plt
import matplotlib as mpl

# DDECC paper (ICLR 2023), N=6, d=128 — Table 1 (line search) and Appendix E Table 4 (Δ = LS − regular)
PAPER_LS = {("BCH",63,36):[5.11,7.09,9.82], ("BCH",63,45):[5.61,7.94,11.36], ("BCH",63,51):[5.26,7.40,10.49],
            ("POLAR",64,32):[6.93,9.51,12.79], ("POLAR",64,48):[5.96,8.04,10.98], ("POLAR",128,64):[9.11,12.90,16.30],
            ("POLAR",128,86):[7.60,10.81,15.17], ("POLAR",128,96):[7.16,10.30,13.19], ("LDPC",49,24):[5.88,8.27,11.42],
            ("LDPC",121,60):[5.38,8.73,14.17], ("LDPC",121,70):[6.79,11.13,16.93], ("LDPC",121,80):[7.59,12.17,16.89],
            ("MACKAY",96,48):[8.12,11.88,15.93], ("CCSDS",128,64):[7.81,12.48,17.66]}
PAPER_DELTA = {("BCH",63,36):[-0.11,-0.02,0.04], ("BCH",63,45):[-0.13,-0.06,0.73], ("BCH",63,51):[0.02,0.34,1.04],
               ("POLAR",64,32):[-0.11,0.07,0.65], ("POLAR",64,48):[-0.01,0.07,0.71], ("POLAR",128,64):[-0.12,-0.20,1.00],
               ("POLAR",128,86):[-0.32,-0.21,-0.40], ("POLAR",128,96):[-0.15,0.00,-0.13], ("LDPC",49,24):[-0.12,-0.13,-0.41],
               ("LDPC",121,60):[-0.18,-0.47,-0.23], ("LDPC",121,70):[-0.18,0.00,-0.11], ("LDPC",121,80):[-0.28,-0.21,-0.27],
               ("MACKAY",96,48):[-0.19,-0.33,-0.13], ("CCSDS",128,64):[-0.26,-0.11,0.37]}
PAPER = {"LS": PAPER_LS, "reg": {c: [round(l - d, 2) for l, d in zip(PAPER_LS[c], PAPER_DELTA[c])] for c in PAPER_LS}}
PE = [4, 5, 6]

S = check(verbose=False)
INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
mpl.rcParams.update({"font.size": 9, "axes.edgecolor": INK2, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": GRID, "legend.frameon": False})
for mode in TEST_MODES:
    print("\n" + "=" * 78 + f"\n{MODE_NAME[mode]}: reproduction − paper (N=6, d=128), −ln(BER)")
    print(f"{'code':<16}" + "".join(f"{e} dB".rjust(22) for e in PE))
    fig, axes = plt.subplots(2, 7, figsize=(20, 6.2), constrained_layout=True)
    for ax, c in zip(axes.flat, CODES):
        paper = PAPER[mode].get(c)
        ours = [S[c]["res"].get((mode, e)) for e in PE]
        cells = ""
        for e, p, r in zip(PE, paper or [None] * 3, ours):
            cells += ("—" if r is None or p is None else f"{lnb(r):.2f} vs {p:.2f} ({lnb(r) - p:+.2f})").rjust(22)
        print(f"{name(c):<16}{cells}")
        if paper:
            ax.plot(PE, paper, color="#eb6834", ls="--", marker="s", lw=2, ms=6, label=f"DDECC paper ({mode}, N=6, d=128)")
        pts = [(e, lnb(r)) for e, r in zip(PE, ours) if r and r["ber"] > 0]
        if pts:
            ax.plot([e for e, _ in pts], [v for _, v in pts], color="#2a78d6", marker="o", lw=2, ms=6,
                    label=f"reproduction ({mode})")
        if paper and len(pts) == 3:
            ax.text(0.04, 0.93, "Δ " + " / ".join(f"{v - p:+.2f}" for (_, v), p in zip(pts, paper)),
                    transform=ax.transAxes, color=INK2, fontsize=8, va="top")
        ax.set_title(name(c), fontsize=10, loc="left", color=INK, fontweight="bold")
        ax.set_xticks(PE); ax.set_xlim(3.8, 6.2)
    for ax in axes[1]:
        ax.set_xlabel("Eb/N0 (dB)")
    for ax in axes[:, 0]:
        ax.set_ylabel("−ln(BER)")
    h, l = axes.flat[0].get_legend_handles_labels()
    fig.legend(h, l, loc="upper right", ncol=2, fontsize=10)
    src = "Table 1" if mode == "LS" else "Table 1 − Δ (Appendix E, Table 4)"
    fig.suptitle(f"{MODE_NAME[mode]}: DDECC reproduction vs. DDECC paper ({src}); Δ = reproduction − paper at 4/5/6 dB",
                 x=0.01, ha="left", fontsize=12)
    fig.savefig(FIGS / f"vs_paper_{mode}.png", dpi=300, bbox_inches="tight")
    plt.show()
print("\nsaved to", FIGS)